# Stage1 MViTv2-S 재학습 (민서 stage1_aihub 데이터, Colab GPU 전용)

**목적**: 기존 MViT는 DACON 공개 10쌍으로만 학습돼 LOO 0/10(페어 암기, 역방향 예측) -
민서가 공유한 AIHub 기반 원본/재녹화 1500쌍(train 2284 / val 716)으로 다시 학습해서
암기가 아닌 일반화된 시각 신호를 얻는 게 목표.

**바꾸지 않는 것**: bpp 임계값(`bpp_thr`/`bpp_scale`)과 블렌드 가중치(`bpp_weight=0.75`)는
DACON 자체 데이터 + 실제 제출 A/B로 이미 검증된 값이라 그대로 유지한다(1.0으로 올렸다가
실제 점수 0.589->0.578로 떨어진 전례 있음, `fit_stage1` docstring 참고). 이 노트북은
**`model` 가중치만** 새로 학습해서 기존 체크포인트 포맷에 그대로 끼워 넣는다.

**절차**:
1. `런타임 > 런타임 유형 변경 > GPU`로 바꾼 뒤 위에서부터 순서대로 실행
2. `stage1_aihub-....zip`을 본인 Google Drive에 올리고 아래 `DRIVE_ZIP_PATH`를 맞게 수정
3. 끝까지 실행하면 `best.pt`가 다운로드됨 - 로컬 `model/stage1/best.pt`로 교체 후
   추론 노트북 5~9단계(`inference.py`/`submit.zip` 재생성)를 다시 돌리면 됨
4. **제출 전 스모크 테스트 + real A/B 없이 바로 제출하지 말 것** - 이 세션의 반복된 교훈
   (UFLD, Stage2 트래킹 둘 다 로컬 검증은 좋았지만 실제 제출에서 회귀)


In [ ]:
# GPU 확인 - 반드시 GPU 런타임에서 실행
!nvidia-smi
import torch
print('torch', torch.__version__, 'cuda available:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'GPU 런타임이 아닙니다. 런타임 > 런타임 유형 변경 > GPU'


In [ ]:
# Google Drive 마운트 + 데이터셋 압축 해제
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ZIP_PATH = '/content/drive/MyDrive/stage1_aihub-20260914T133104Z-1-001.zip'  # 본인 경로로 수정

import os
assert os.path.isfile(DRIVE_ZIP_PATH), f'파일을 찾을 수 없음: {DRIVE_ZIP_PATH} - Drive 업로드 경로를 확인하세요.'

!rm -rf /content/stage1_aihub
!mkdir -p /content/stage1_aihub
!unzip -q "$DRIVE_ZIP_PATH" -d /content/stage1_aihub
!find /content/stage1_aihub -maxdepth 2 -type d | sort


In [ ]:
# 매니페스트 로드 - train/val split, ORIGINAL/RERECORDED 라벨
import csv
from pathlib import Path

ROOT = Path('/content/stage1_aihub/stage1_aihub')
manifest_path = ROOT / 'stage1_manifest.csv'
rows = list(csv.DictReader(open(manifest_path, encoding='utf-8')))
rows = [r for r in rows if Path(ROOT.parent / r['path'].replace('train_data/', '')).is_file() or
        (ROOT / r['path'].split('stage1_aihub/', 1)[-1]).is_file()]

def resolve(path_field):
    # manifest의 path는 'train_data/stage1_aihub/...' 접두어 - 실제 압축 해제 경로로 맞춰줌
    rel = path_field.split('stage1_aihub/', 1)[-1]
    return ROOT / rel

train_rows = [r for r in rows if r['split'] == 'train']
val_rows = [r for r in rows if r['split'] == 'val']
print('train', len(train_rows), 'val', len(val_rows))
print(resolve(train_rows[0]['path']), resolve(train_rows[0]['path']).is_file())


In [ ]:
# 기존 프로덕션과 동일한 클립 디코딩/모델 정의 (src/stage1 학습 코드와 동일 스펙)
import random
import numpy as np
import cv2
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision.models.video import mvit_v2_s, MViT_V2_S_Weights

SIZE, FRAMES = 224, 16
S1_MEAN = torch.tensor([0.45, 0.45, 0.45])[:, None, None, None]
S1_STD = torch.tensor([0.225, 0.225, 0.225])[:, None, None, None]


def _crop(rgb, size=SIZE):
    h, w = rgb.shape[:2]
    scale = size / min(h, w)
    nh, nw = max(size, round(h * scale)), max(size, round(w * scale))
    rgb = cv2.resize(rgb, (nw, nh), interpolation=cv2.INTER_AREA)
    y, x = (nh - size) // 2, (nw - size) // 2
    return rgb[y:y + size, x:x + size]


class Stage1Dataset(Dataset):
    """영상 전체를 메모리에 올리지 않고 필요한 프레임만 디코딩. 매 에폭 다른 시작
    오프셋 + 좌우반전 + 밝기 지터로 암기 방지(2284개로도 ViT는 쉽게 외울 수 있음)."""

    def __init__(self, rows, train: bool):
        self.rows = rows
        self.train = train

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, index):
        row = self.rows[index]
        path = resolve(row['path'])
        label = 0 if row['label'] == 'ORIGINAL' else 1
        cap = cv2.VideoCapture(str(path))
        total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)) or 1
        rng = random.Random(hash((path.name, self.train)) & 0xFFFF) if not self.train else random
        if self.train and total > FRAMES:
            start = random.randint(0, total - FRAMES)
        else:
            start = max(0, (total - FRAMES) // 2)
        wanted = set(range(start, min(total, start + FRAMES)))
        flip = self.train and random.random() < 0.5
        gain = random.uniform(0.85, 1.15) if self.train else 1.0
        frames = []
        idx = 0
        while cap.isOpened() and len(frames) < FRAMES:
            ok, bgr = cap.read()
            if not ok:
                break
            if idx in wanted:
                rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
                rgb = _crop(rgb)
                if flip:
                    rgb = rgb[:, ::-1]
                if gain != 1.0:
                    rgb = np.clip(rgb.astype(np.float32) * gain, 0, 255).astype(np.uint8)
                frames.append(rgb)
            idx += 1
        cap.release()
        if not frames:
            frames = [np.zeros((SIZE, SIZE, 3), dtype=np.uint8)]
        while len(frames) < FRAMES:
            frames.append(frames[-1])
        x = torch.from_numpy(np.stack(frames)).permute(3, 0, 1, 2).float() / 255.0
        x = (x - S1_MEAN) / S1_STD
        return x, label


class Stage1MViT(nn.Module):
    def __init__(self, pretrained=True):
        super().__init__()
        weights = MViT_V2_S_Weights.KINETICS400_V1 if pretrained else None
        self.net = mvit_v2_s(weights=weights)
        self.net.head[1] = nn.Linear(self.net.head[1].in_features, 2)

    def forward(self, x):
        return self.net(x)


In [ ]:
# 학습 - 사전학습 가중치(Kinetics400)에서 시작, val 정확도 최고점에서 체크포인트 저장
EPOCHS = 6
BATCH_SIZE = 8
LR = 3e-5  # 사전학습 파인튜닝이라 예전(1e-4, from-scratch)보다 낮춤

device = torch.device('cuda')
model = Stage1MViT(pretrained=True).to(device)
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)

train_loader = DataLoader(Stage1Dataset(train_rows, train=True), batch_size=BATCH_SIZE,
                           shuffle=True, num_workers=2, drop_last=True)
val_loader = DataLoader(Stage1Dataset(val_rows, train=False), batch_size=BATCH_SIZE,
                         shuffle=False, num_workers=2)

def run_epoch(loader, train: bool):
    model.train(train)
    total = correct = 0
    loss_sum = 0.0
    for x, y in loader:
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        with torch.set_grad_enabled(train):
            with torch.autocast(device_type='cuda', dtype=torch.float16):
                logits = model(x)
                loss = nn.functional.cross_entropy(logits, y)
            if train:
                opt.zero_grad()
                loss.backward()
                opt.step()
        loss_sum += float(loss) * len(y)
        correct += int((logits.argmax(1) == y).sum())
        total += len(y)
    return loss_sum / total, correct / total

best_val_acc = -1.0
best_state = None
for epoch in range(1, EPOCHS + 1):
    train_loss, train_acc = run_epoch(train_loader, train=True)
    val_loss, val_acc = run_epoch(val_loader, train=False)
    print(f'epoch {epoch}/{EPOCHS}  train_loss={train_loss:.4f} train_acc={train_acc:.3f}  '
          f'val_loss={val_loss:.4f} val_acc={val_acc:.3f}')
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_state = {k: v.detach().cpu().clone() for k, v in model.net.state_dict().items()}
        print(f'  -> new best (val_acc={best_val_acc:.3f}), 체크포인트 후보 갱신')

print('최종 최고 val_acc:', best_val_acc)
assert best_state is not None


In [ ]:
# 기존 프로덕션 체크포인트 포맷 그대로 저장 - bpp_thr/scale/weight은 DACON 실측 A/B로
# 이미 검증된 값이라 하드코딩 유지(재도출 안 함). model만 새 가중치로 교체.
OUT_PATH = '/content/best.pt'
torch.save({
    'model': best_state,
    'size': SIZE,
    'frames': FRAMES,
    'bpp_thr': 0.09921241319444445,
    'bpp_scale': 0.007743836805555557,
    'bpp_weight': 0.75,
}, OUT_PATH)
print('저장 완료:', OUT_PATH, ' best_val_acc=%.3f' % best_val_acc)

from google.colab import files
files.download(OUT_PATH)


## 다음 단계 (로컬)

1. 다운로드된 `best.pt`를 `model/stage1/best.pt`로 교체
2. 추론 노트북(`[Baseline_Inference]_3Stage_추론및ZIP생성.ipynb`) 5~9단계 재실행 -
   `inference.py`/`submit.zip` 재생성, 공개 5샘플 스모크 테스트
3. **바로 제출하지 말고** 위 val_acc가 기존(=MViT 노이즈 수준, ~50%)보다 확실히
   높은지 먼저 공유 - 이번에도 로컬 개선이 실제 제출과 어긋날 수 있음(이 세션에서
   이미 2번 반복된 패턴)
